In [1]:
%load_ext autoreload
%autoreload 2

from cgnn.dataloader import HospitalizationAdvanPlusDatasetLoader
from torch_geometric_temporal.signal import temporal_signal_split
from omegaconf import OmegaConf

# Load config with dates from hospital_advan.yaml
cfg = OmegaConf.load("../experiments/conf/data_dates/hospital_advan.yaml")
cfg.data.end_date = '12/31/2021'

# Initialize loader with config
loader = HospitalizationAdvanPlusDatasetLoader(cbsa_list=None, cfg=cfg, transform_xy=True)

# Get dataset
dataset = loader.get_dataset()

# Split into train/test
train_dataset, test_dataset = temporal_signal_split(dataset, train_ratio=0.8)

dates = [d.strftime("%Y-%m-%d") for d in loader.dates]
train_size = train_dataset.snapshot_count
train_dates = dates[:train_size]
test_dates = dates[train_size:]

/Users/hwunrow/tgt/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading hospitalization data...


/Users/hwunrow/Documents/GitHub/cgnn/src/cgnn/process_data.py:1001: DtypeWarning: Columns (0,3) have mixed types. Specify dtype option on import or set low_memory=False.
  hosp_df = pd.read_csv(raw_hospitalization_file, dtype={"fips_code": str})


number of hospitals with less than max_missing_weeks=21 missing weeks: 4978
unique ZIPs in zip-cbsa map: 39502
unique ZIPs in zip-county map: 39490
num zips in cbsa but not county: 15
num zips in county but not cbsa: 3
Loading Advan Plus mobility data...
Creating temporal snapshots...


Processing snapshots: 100%|█████████████████████████████████████████████████████████████████████████████████| 77/77 [02:40<00:00,  2.08s/it]

Created 77 snapshots
Number of nodes per snapshot: 918
Average edges per snapshot: 39618.0


In [2]:
cfg

{'data': {'start_date': '07/13/2020', 'end_date': '12/31/2021'}, 'model': {'node_features': 1, 'target_feature_idx': 0}}

In [3]:
import torch
torch.manual_seed(1)
import torch.nn.functional as F
from torch_geometric_temporal.nn.recurrent import DCRNN

class RecurrentGCN(torch.nn.Module):
    def __init__(self, node_features):
        super(RecurrentGCN, self).__init__()
        self.recurrent = DCRNN(node_features, 33, 1)
        self.linear = torch.nn.Linear(33, 1)
        # self.residual = torch.nn.Linear(node_features, 1)

    def forward(self, x, edge_index, edge_weight):
        h = self.recurrent(x, edge_index, edge_weight)
        h = F.relu(h)
        h = self.linear(h)

        return h
        # h_skip = self.residual(x)
        
        # return h + h_skip

In [ ]:
from tqdm import tqdm

model = RecurrentGCN(node_features=1)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=1e-4)

train_losses = []
test_losses = []

for epoch in tqdm(range(300)):
    model.train()
    train_cost = 0.0
    for time, snapshot in enumerate(train_dataset):
        y_hat = model(snapshot.x, snapshot.edge_index, snapshot.edge_attr)
        pred_linear = torch.expm1(y_hat.squeeze())
        target_linear = torch.expm1(snapshot.y)
        train_cost = train_cost + torch.mean((pred_linear - target_linear) ** 2)
    
    train_cost = train_cost / (time + 1)
    train_cost.backward()
    optimizer.step()
    optimizer.zero_grad()

    # Track train loss
    train_losses.append(train_cost.item())

    # Evaluate on test set each epoch
    model.eval()
    with torch.no_grad():
        test_cost = 0.0
        for t, snapshot in enumerate(test_dataset):
            y_hat = model(snapshot.x, snapshot.edge_index, snapshot.edge_attr)
            pred_linear = torch.expm1(y_hat.squeeze())
            target_linear = torch.expm1(snapshot.y)
            test_cost = test_cost + torch.mean((pred_linear - target_linear) ** 2)
        test_cost = test_cost / (t + 1)
        test_losses.append(test_cost.item())

    # Optional: print every 50 epochs
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}: train MSE={train_cost.item():.4f}, test MSE={test_cost.item():.4f}")

  9%|█████████▏                                                                                            | 27/300 [00:13<02:11,  2.08it/s]

In [ ]:
model.residual.weight

In [ ]:
model.residual.bias

In [ ]:
print(model.recurrent.conv_x_h.weight.mean())
print(model.recurrent.conv_x_h.weight.std())

In [ ]:
print(model.recurrent.conv_x_z.weight.mean())
print(model.recurrent.conv_x_z.weight.std())

In [ ]:
print(model.recurrent.conv_x_r.weight.mean())
print(model.recurrent.conv_x_r.weight.std())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="Train MSE")
plt.plot(test_losses, label="Test MSE")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Training vs Test Loss")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


def evaluate_and_plot(model, dataset, device, dataset_name="Dataset", top_nodes=None, dates=None, offset=0):
    """Plot with optional date labels; offset indexes into the global dates list."""
    model.eval()
    cost = 0
    all_preds = []
    all_truth = []
    
    with torch.no_grad():
        for time, snapshot in enumerate(dataset):
            snapshot.to(device)
            y_hat = model(snapshot.x, snapshot.edge_index, snapshot.edge_attr)
            pred = y_hat.squeeze()
            y = snapshot.y
            pred = torch.expm1(pred)
            y = torch.expm1(y)

            cost += torch.mean((pred - y) ** 2)
            
            all_preds.append(pred.cpu().numpy())
            all_truth.append(y.cpu().numpy())
    
    cost = cost / (time + 1)
    print(f"{dataset_name} MSE: {cost.item():.4f}")
    
    pred_matrix = np.vstack(all_preds)
    truth_matrix = np.vstack(all_truth)

    if dataset_name == "Test":
        truth_matrix, pred_matrix = truth_matrix[:-1], pred_matrix[:-1]    

    print(pred_matrix.shape)
    print(truth_matrix.shape)
    
    if top_nodes is None:
        means = np.mean(truth_matrix, axis=0)
        top_nodes = np.argsort(means)[-3:]
    
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams.update({
        'font.family': 'serif',
        'font.size': 12,
        'axes.labelsize': 14,
        'axes.titlesize': 16,
        'xtick.labelsize': 12,
        'ytick.labelsize': 12,
        'legend.fontsize': 12,
        'figure.dpi': 300,
        'lines.linewidth': 2
    })
    
    mean_pred = pred_matrix.sum(axis=1)
    mean_truth = truth_matrix.sum(axis=1)
    time_steps = np.arange(len(mean_pred) - 1)

    print(np.mean((mean_pred - mean_truth)**2))
    
    # Date labels aligned with one-step-ahead shift (pred[1:] vs truth[:-1])
    if dates is not None:
        slice_dates = dates[offset : offset + len(mean_pred)]
        # date_labels = slice_dates[1:]
        date_labels = slice_dates
        date_labels = pd.to_datetime(date_labels)
    else:
        date_labels = time_steps

    plt.figure(figsize=(10, 6))
    plt.plot(truth_matrix, pred_matrix, '.', color='gray')
    plt.axline((0, 0), slope=1., color='red')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.xlabel('truth')
    plt.ylabel('prediction')
    plt.title(f'{dataset_name}')
    plt.show()
    
    plt.figure(figsize=(10, 6))
    plt.plot(date_labels, mean_truth, label='Ground Truth', color='#333333', alpha=0.8, linestyle='-')
    plt.plot(date_labels, mean_pred, label='Model Prediction', color='#E24A33', linestyle='--')
    plt.fill_between(date_labels, mean_truth, mean_pred, color='gray', alpha=0.1, label='Error Gap')
    # plt.plot(date_labels, mean_truth[:-1], label='Ground Truth', color='#333333', alpha=0.8, linestyle='-')
    # plt.plot(date_labels, mean_pred[1:], label='Model Prediction', color='#E24A33', linestyle='--')
    # plt.fill_between(date_labels, mean_truth[:-1], mean_pred[1:], color='gray', alpha=0.1, label='Error Gap')
    plt.xlabel("Date" if dates is not None else "Time Steps (Days)")
    plt.ylabel("Hospitalizations")
    plt.title(f"National Forecast Performance - {dataset_name}")
    plt.legend(frameon=True, framealpha=0.9, loc='upper right')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.show()
    
    fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharex=True)
    for i, node_idx in enumerate(top_nodes):
        ax = axes[i]
        ax.plot(date_labels, truth_matrix[:, node_idx], label='Ground Truth', color='black', alpha=0.7)
        ax.plot(date_labels, pred_matrix[:, node_idx], label='Prediction', color='#348ABD', linestyle='--')
        ax.set_ylabel(f"Node {node_idx}\nValues")
        ax.set_title(f"Forecast for Region #{node_idx}")
        ax.grid(True, linestyle=':', alpha=0.6)
        if i == 0:
            ax.legend(loc='upper right')

        print(np.mean((truth_matrix[:-1, node_idx] - pred_matrix[1:, node_idx])**2))
    
    plt.xlabel("Date" if dates is not None else "Time Steps (Days)")
    plt.suptitle(f"Local Forecast Performance - {dataset_name} (Top Active Regions)", y=1.02, fontsize=16)
    plt.tight_layout()
    plt.show()
    
    return top_nodes

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dates = [d.strftime('%Y-%m-%d') for d in loader.dates]
train_size = train_dataset.snapshot_count

top_nodes = evaluate_and_plot(model, train_dataset, device, "Train", dates=dates, offset=0)
top_nodes = evaluate_and_plot(model, test_dataset, device, "Test", top_nodes=top_nodes, dates=dates, offset=train_size)


In [ ]:
import torch
from torch.optim import Adam
import matplotlib.pyplot as plt
import networkx as nx

class DCRNNExplainer:
    def __init__(self, model, device):
        self.model = model
        self.device = device
        self.model.eval() # Important: Freeze the model weights

    def explain(self, x, edge_index, edge_weight=None, epochs=500, lr=0.01):
        """
        Explains a single snapshot prediction by learning an edge mask.
        
        Args:
            x: Node features for the snapshot (Num_Nodes, Num_Features)
            edge_index: Graph connectivity (2, Num_Edges)
            edge_weight: Original edge weights (Optional)
        """
        num_edges = edge_index.shape[1]
        
        # 1. Initialize Learnable Mask (Logits)
        # We start with random values; the sigmoid will map them to [0,1]
        mask_logits = torch.randn(num_edges, requires_grad=True, device=self.device)
        
        # Handle existing weights (default to 1.0 if None)
        if edge_weight is None:
            base_weights = torch.ones(num_edges, device=self.device)
        else:
            base_weights = edge_weight.to(self.device)

        optimizer = Adam([mask_logits], lr=lr)

        # 2. Get Baseline Prediction (Original Graph)
        with torch.no_grad():
            base_pred = self.model(x, edge_index, base_weights)
            base_pred = torch.expm1(base_pred)

        with torch.no_grad():
            zero_x_pred = model(torch.zeros_like(x), edge_index, edge_weight)
            zero_x_pred = torch.expm1(zero_x_pred)

        print(f"Explaining snapshot with {num_edges} edges...")

        # 3. Optimization Loop
        for epoch in range(epochs):
            optimizer.zero_grad()
            
            # Sigmoid converts logits to a mask in [0, 1]
            mask = torch.sigmoid(mask_logits)
            
            # Apply mask to the original weights
            # This "soft deletes" edges where mask is near 0
            masked_weights = base_weights * mask
            
            # Forward pass with masked graph
            # Note: We must detach the model parameters to only train the mask
            # masked_weights = base_weights * torch.zeros(base_weights.shape)
            masked_pred = self.model(x, edge_index, masked_weights)
            masked_pred = torch.expm1(masked_pred)
            
            # --- Loss Calculation ---
            # A. Fidelity: Prediction should stay close to the original
            loss_dist = torch.mean((masked_pred - base_pred) ** 2)
            
            # B. Sparsity: Encourage mask sum to be small (remove irrelevant edges)
            loss_size = 0.00005 * mask.sum()
            
            # C. Entropy: Encourage mask values to be binary (0 or 1, not 0.5)
            # This makes the subgraph "sharp" rather than fuzzy
            ent = -mask * torch.log(mask + 1e-15) - (1 - mask) * torch.log(1 - mask + 1e-15)
            loss_ent = 0.1 * ent.mean()

            loss = loss_dist + loss_size + loss_ent
            
            loss.backward()
            optimizer.step()
            
        # Return the learned mask (detached from graph)
        print(f"{loss_dist=}")
        print(f"{loss_size=}")
        print(f"{loss_ent=}")
        print(f"{loss=}")
        return torch.sigmoid(mask_logits).detach()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

num_explainable_edges_list = []
for i, snapshot in tqdm(enumerate(train_dataset)):
    x_snapshot = snapshot.x.to(device)
    edge_index_snapshot = snapshot.edge_index.to(device)
    edge_weight_snapshot = snapshot.edge_attr.to(device) if snapshot.edge_attr is not None else None

    explainer = DCRNNExplainer(model, device)

    importance_mask = explainer.explain(x_snapshot, 
                                        edge_index_snapshot, 
                                        edge_weight_snapshot, 
                                        epochs=200, 
                                        lr=0.05)
    num_explainable_edges = torch.sum(importance_mask > 0.5).item()
    num_explainable_edges_list.append(num_explainable_edges)
    if num_explainable_edges > 0:
        print(f"Snapshot {i} has {num_explainable_edges} explainable edges")


# Collect ground-truth totals (sum over nodes) per snapshot
truth_totals = []
for snap in train_dataset:
    truth_totals.append(snap.y.sum().item())

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(num_explainable_edges_list, color="#E24A33", label="Explainable edges")
ax1.set_xlabel("Time Step")
ax1.set_ylabel("Explainable edges", color="#E24A33")
ax1.tick_params(axis="y", labelcolor="#E24A33")

ax2 = ax1.twinx()
ax2.plot(truth_totals[:-1], color="#333333", linestyle="--", label="Ground Truth")
ax2.set_ylabel("Ground truth sum", color="#333333")
ax2.tick_params(axis="y", labelcolor="#333333")

fig.suptitle("Explainable edges vs. Ground Truth (Train)")
fig.tight_layout()
fig.legend(loc="upper right")
plt.grid(True, linestyle=":", alpha=0.6)
plt.show()

In [26]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

num_explainable_edges_list = []
for i, snapshot in enumerate(test_dataset):
    x_snapshot = snapshot.x.to(device)
    edge_index_snapshot = snapshot.edge_index.to(device)
    edge_weight_snapshot = snapshot.edge_attr.to(device) if snapshot.edge_attr is not None else None

    explainer = DCRNNExplainer(model, device)

    # Get importance mask for this specific day
    importance_mask = explainer.explain(x_snapshot, 
                                        edge_index_snapshot, 
                                        edge_weight_snapshot, 
                                        epochs=200, 
                                        lr=0.01)
    num_explainable_edges = torch.sum(importance_mask > 0.5).item()
    num_explainable_edges_list.append(num_explainable_edges)
    if num_explainable_edges > 0:
        print(f"Snapshot {i} has {num_explainable_edges} explainable edges")


Explaining snapshot with 40627 edges...
loss_dist=tensor(0.0909, grad_fn=<MeanBackward0>)
loss_size=tensor(0.3315, grad_fn=<MulBackward0>)
loss_ent=tensor(0.0412, grad_fn=<MulBackward0>)
loss=tensor(0.4637, grad_fn=<AddBackward0>)
Snapshot 0 has 165 explainable edges
Explaining snapshot with 40782 edges...


KeyboardInterrupt: 

In [ ]:
# Collect ground-truth totals (sum over nodes) per snapshot
truth_totals = []
for snap in test_dataset:
    truth_totals.append(snap.y.sum().item())

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(num_explainable_edges_list, color="#E24A33", label="Explainable edges")
ax1.set_xlabel("Time Step")
ax1.set_ylabel("Explainable edges", color="#E24A33")
ax1.tick_params(axis="y", labelcolor="#E24A33")

ax2 = ax1.twinx()
ax2.plot(truth_totals[:-1], color="#333333", linestyle="--", label="Ground Truth")
ax2.set_ylabel("Ground truth sum", color="#333333")
ax2.tick_params(axis="y", labelcolor="#333333")

fig.suptitle("Explainable edges vs. Ground Truth (Test)")
fig.tight_layout()
fig.legend(loc="upper right")
plt.grid(True, linestyle=":", alpha=0.6)
plt.show()

## EvolveGCNH

In [27]:
import torch
import torch.nn.functional as F
from torch_geometric_temporal.nn.recurrent import EvolveGCNH

class CovidEvolveModel(torch.nn.Module):
    def __init__(self, node_count, node_features):
        super(CovidEvolveModel, self).__init__()
        
        # EvolveGCNH: "Evolving Graph Convolutional Network (Hybrid)"
        # It adapts the GCN weights based on the changing graph structure
        self.recurrent = EvolveGCNH(num_of_nodes=node_count, 
                                    in_channels=node_features)
        
        # Final layer to map embedding -> Hospitalization Count
        self.linear = torch.nn.Linear(node_features, 1)

    def forward(self, x, edge_index, edge_weight=None):
        # 1. Evolve weights & Convolve
        # Note: EvolveGCN maintains internal state (hidden weights) automatically
        h = self.recurrent(x, edge_index, edge_weight)
        
        # 2. Activation
        h = F.relu(h)
        
        # 3. Final Prediction
        h = self.linear(h)
        return h

In [40]:
from tqdm import tqdm

# --- Hyperparameters ---
LEARNING_RATE = 0.005
EPOCHS = 50
ACCUMULATION_STEPS = 7  # Update weights every 7 days (Weekly batching)

# --- Setup ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Assuming 'num_nodes' is 20 and 'num_features' is 1 based on previous context
model = CovidEvolveModel(node_count=129, node_features=8).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
model.train()

print(f"Starting training on device: {device}")

# --- Training Loop ---
for epoch in range(EPOCHS):
    total_loss = 0
    step = 0
    optimizer.zero_grad()
    
    # The train_dataset is an iterator yielding (x, edge_index, y) for each time step
    # We wrap it in tqdm for a progress bar
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    
    for time_step, snapshot in enumerate(train_dataset):
        # Move data to GPU/CPU
        snapshot.to(device)
        
        # 1. Forward Pass
        # EvolveGCN takes the specific edge_index for THIS time step
        y_hat = model(snapshot.x, snapshot.edge_index, snapshot.edge_attr)
        
        # 2. Calculate Loss
        # We assume snapshot.y is the target for the nodes at this step
        # y_hat shape: (Nodes, 1) -> Squeeze to match snapshot.y
        loss = torch.mean((y_hat.squeeze() - snapshot.y) ** 2)
        
        # 3. Backward (Accumulate Gradients)
        # We normalize by accumulation steps to keep gradient magnitude consistent
        (loss / ACCUMULATION_STEPS).backward(retain_graph=True)
        
        total_loss += loss.item()
        step += 1
        
        # 4. Optimizer Step (Truncated BPTT)
        if step % ACCUMULATION_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()
            
            # CRITICAL FOR EVOLVEGCN:
            # EvolveGCN stores the GCN weights as a hidden state.
            # If we don't detach, PyTorch tries to backpropagate all the way 
            # to the start of time (Day 0), causing Out-Of-Memory errors.
            model.recurrent.weight = model.recurrent.weight.detach()

    # Average loss for the epoch
    avg_loss = total_loss / (time_step + 1)
    print(f"Average MSE Loss: {avg_loss:.4f}")

print("Training Complete.")

Starting training on device: cpu

Epoch 1/50
Average MSE Loss: 1.3911

Epoch 2/50
Average MSE Loss: 0.8615

Epoch 3/50
Average MSE Loss: 0.5482

Epoch 4/50
Average MSE Loss: 0.3976

Epoch 5/50
Average MSE Loss: 0.3711

Epoch 6/50
Average MSE Loss: 0.4030

Epoch 7/50
Average MSE Loss: 0.4287

Epoch 8/50
Average MSE Loss: 0.4398

Epoch 9/50
Average MSE Loss: 0.4188

Epoch 10/50
Average MSE Loss: 0.3978

Epoch 11/50
Average MSE Loss: 0.4050

Epoch 12/50
Average MSE Loss: 0.4334

Epoch 13/50
Average MSE Loss: 0.4317

Epoch 14/50
Average MSE Loss: 0.4112

Epoch 15/50
Average MSE Loss: 0.3921

Epoch 16/50
Average MSE Loss: 0.3844

Epoch 17/50
Average MSE Loss: 0.3891

Epoch 18/50
Average MSE Loss: 0.3933

Epoch 19/50
Average MSE Loss: 0.3899

Epoch 20/50
Average MSE Loss: 0.3846

Epoch 21/50
Average MSE Loss: 0.3825

Epoch 22/50
Average MSE Loss: 0.3839

Epoch 23/50
Average MSE Loss: 0.3907

Epoch 24/50
Average MSE Loss: 0.3923

Epoch 25/50
Average MSE Loss: 0.3891

Epoch 26/50
Average MSE L

In [41]:
model.eval()
cost = 0
for time, snapshot in enumerate(test_dataset):
    y_hat = model(snapshot.x, snapshot.edge_index, snapshot.edge_attr)
    cost = cost + torch.mean((y_hat.squeeze() - snapshot.y)**2)
cost = cost / (time+1)
cost = cost.item()
print("MSE: {:.4f}".format(cost))

MSE: 1.0966
